### ЗАДАЧА: Пакетная обработка переводов между кошельками (exceptions + business rules)

Есть набор кошельков и список входящих переводов.
Нужно безопасно обработать пакет операций: валидные переводы применить к балансам,
ошибочные сохранить в отчёт, не останавливая всю обработку.

НЕОБХОДИМО РЕАЛИЗОВАТЬ:

1. Иерархию кастомных исключений:
   - `TransferError`
   - `TransferFormatError`
   - `AccountNotFoundError`
   - `CurrencyMismatchError`
   - `InsufficientFundsError`
   - `TransferAmountError`.

2. Функцию `parse_transfer(raw)`:
   - формат строки: `transfer_id|from_user|to_user|amount`
   - `amount` должен быть числом и `> 0`
   - при ошибке конвертации использовать `raise ... from ...`.

3. Функцию `apply_transfer(transfer, wallets)`:
   - проверить, что оба пользователя существуют
   - нельзя переводить самому себе
   - валюты кошельков отправителя и получателя должны совпадать
   - у отправителя должно хватать средств
   - при успехе обновить балансы в `wallets`
   - вернуть краткий словарь результата.

4. Функцию `process_batch(rows, wallets)`:
   - для каждой строки вызвать `parse_transfer`, потом `apply_transfer`
   - вернуть `(successes, errors)`
   - ошибки хранить как `(raw, error_type, message)`
   - не прерывать цикл на первой ошибке.

5. Вывести:
   - успешные переводы,
   - ошибки по типам,
   - итоговые балансы,
   - пользователя с максимальным балансом в валюте `USD`.


In [ ]:
wallets = {
    'alice': {'currency': 'USD', 'balance': 1200.0},
    'bob': {'currency': 'USD', 'balance': 450.0},
    'carol': {'currency': 'EUR', 'balance': 900.0},
    'dave': {'currency': 'USD', 'balance': 150.0},
}

rows = [
    'TR-100|alice|bob|200',
    'TR-101|bob|dave|700',
    'TR-102|alice|carol|50',
    'TR-103|eve|bob|30',
    'TR-104|dave|dave|10',
    'TR-105|bob|alice|abc',
    'TR-106|bob|dave|100',
]

class TransferError(Exception):
    pass

class TransferFormatError(TransferError):
    pass

class AccountNotFoundError(TransferError):
    pass

class CurrencyMismatchError(TransferError):
    pass

class InsufficientFundsError(TransferError):
    pass

class TransferAmountError(TransferError):
    pass

def parse_transfer(raw):
    # TODO: распарсить строку и вернуть dict перевода
    try:
        parts = raw.split("|")
        if len(parts) != 4:
            raise TransferFormatError(f"Неверный формат передачи: ожидалось 4 поля, получено {len(parts)}")
        
        transfer_id, sender, receiver, amount_str = parts

        try:
            amount = float(amount_str)
            if amount <= 0:
                raise TransferAmountError(f"Сумма перевода должна быть положительной, получено {amount}")
            
        # TODO: при ошибке конвертации amount использовать raise ... from ...
        except ValueError as e:
            raise TransferAmountError(f"Неверное значение суммы: '{amount_str}'") from e
        
        return {
            "transfer_id": transfer_id,
            "sender": sender,
            "receiver": receiver,
            "amount": amount
        }
    
    except TransferError:
        raise
    except Exception as e:
        raise TransferFormatError(f"Неожиданная ошибка при разборе передачи: {e}") from e

def apply_transfer(transfer, wallets):
    sender = transfer["sender"]
    receiver = transfer["receiver"]
    amount = transfer["amount"]

    # TODO: проверить существование аккаунтов
    if sender not in wallets:
        raise AccountNotFoundError(f"Аккаунт отправителя не найден: {sender}")
    if receiver not in wallets:
        raise AccountNotFoundError(f"Счет получателя не найден: {receiver}")

    # TODO: запретить перевод самому себе
    if sender == receiver:
        raise TransferError(f"Невозможно перевести себе: {sender}")

    # TODO: проверить совпадение валют
    sender_currency = wallets[sender]["currency"]
    receiver_currency = wallets[receiver]["currency"]
    if sender_currency != receiver_currency:
        raise CurrencyMismatchError(f"Несоответствие валют: {sender} ({sender_currency}) {receiver} ({receiver_currency})")

    # TODO: проверить баланс отправителя
    if wallets[sender]["balance"] < amount:
        raise InsufficientFundsError(f"Недостаточно средств: у {sender} {wallets[sender]["balance"]}, необходимо {amount}")

    # TODO: обновить балансы и вернуть dict результата
    wallets[sender]["balance"] -= amount
    wallets[receiver]["balance"] += amount

    return {
        "transfer_id": transfer["transfer_id"],
        "sender": sender,
        "receiver": receiver,
        "amount": amount,
        "currency": sender_currency,
    }

def process_batch(rows, wallets):
    # TODO: вернуть (successes, errors)
    successes = []
    errors = []

    for raw in rows:
        try:
            transfer = parse_transfer(raw)
            result = apply_transfer(transfer, wallets)
            successes.append(result)
        except TransferError as e:
            errors.append((raw, type(e).__name__, e))

    return successes, errors

# TODO: вызвать process_batch(rows, wallets)
successes, errors = process_batch(rows, wallets)

# TODO: вывести успешные переводы
for success in successes:
    print(f"{success["transfer_id"]}: {success["sender"]} -> {success["receiver"]} {success["amount"]} {success["currency"]}")

# TODO: вывести ошибки по типам
error_types = {}
for _,error_type,_ in errors:error_types[error_type] = error_types.get(error_type, 0) + 1
for error_type, count in error_types.items():
    print(f"{error_type}: {count}")

# TODO: вывести итоговые балансы
for user, info in wallets.items():
    print(f"{user}: {info["balance"]} {info["currency"]}")

# TODO: найти richest_usd_user
richest_usd_user = None
max_usd_balance = -1

for user, info in wallets.items():
    if info["currency"] == "USD" and info["balance"] > max_usd_balance:
        max_usd_balance = info["balance"]
        richest_usd_user = user

if richest_usd_user:
    print(f"\nRichest USD user: {richest_usd_user} с {max_usd_balance} USD")
else:
    print(f"Пользователи с USD не найдены")